In [8]:
import os
import urllib.request
import warnings

import numpy as np

warnings.filterwarnings("ignore")

SIZES = (4, 4, 4, 1)
EPOCHS, TEST_SIZE, THR = 3000, 0.25, 0.05
SEEDS = range(1, 6)
OPTS = ["adam", "rmsprop", "sgd"]
ACTS = ["silu", "sigmoid", "tanh"]
LR = {"adam": 0.01, "rmsprop": 0.003, "sgd": 0.5}
MAN_LR = {"adam": 0.05, "rmsprop": 0.01, "sgd": 0.5}
MAN_EPOCHS = 200
BETA1, BETA2, RHO, EPS = 0.9, 0.999, 0.9, 1e-8

MAN_INIT = [np.array([[0.21, -0.35, 0.12, 0.40], [0.24, 0.56, -0.18, 0.09],
                      [0.30, -0.15, -0.33, 0.22], [-0.27, 0.41, 0.08, -0.19]]),
            np.array([[0.20, 0.05, -0.05, 0.10]]),
            np.array([[0.48, -0.27, 0.36, 0.14], [0.17, 0.52, -0.41, 0.25],
                      [-0.33, 0.29, 0.44, -0.12], [0.38, -0.16, 0.07, 0.51]]),
            np.array([[0.15, 0.02, -0.08, 0.05]]),
            np.array([[0.60], [-0.45], [0.32], [-0.21]]),
            np.array([[0.10]])]


def load_data(path="banknote.csv"):
    if not os.path.exists(path):
        urllib.request.urlretrieve(
            "https://raw.githubusercontent.com/jbrownlee/Datasets/master/banknote_authentication.csv", path)
    d = np.loadtxt(path, delimiter=",")
    return d[:, :4], d[:, 4:5]


def split(X, y):
    rng = np.random.default_rng(0)
    tr, te = [], []
    for c in np.unique(y):
        i = np.where(y.ravel() == c)[0]
        rng.shuffle(i)
        k = int(round(len(i) * TEST_SIZE))
        te += list(i[:k])
        tr += list(i[k:])
    tr, te = np.array(tr), np.array(te)
    rng.shuffle(tr)
    rng.shuffle(te)
    mu, sd = X[tr].mean(axis=0), X[tr].std(axis=0)
    return (X[tr] - mu) / sd, y[tr], (X[te] - mu) / sd, y[te]


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -60, 60)))


def sigmoid_grad(z, a):
    return a * (1 - a)


def silu(z):
    return z * sigmoid(z)


def silu_grad(z, a):
    s = sigmoid(z)
    return s * (1.0 + z * (1.0 - s))


def tanh(z):
    return 2 * sigmoid(2 * z) - 1


def tanh_grad(z, a):
    return 1 - a ** 2


ACT = {"silu": (silu, silu_grad), "sigmoid": (sigmoid, sigmoid_grad), "tanh": (tanh, tanh_grad)}


def bce(p, y):
    p = np.clip(p, 1e-12, 1 - 1e-12)
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))


def acc(p, y):
    return float(np.mean((p > 0.5) == (y > 0.5)))


def reached(loss):
    i = np.where(np.array(loss) < THR)[0]
    return int(i[0]) + 1 if len(i) else len(loss)


def init_params(act, seed):
    rng = np.random.default_rng(seed)
    P = []
    for n_in, n_out in zip(SIZES[:-1], SIZES[1:]):
        std = np.sqrt(2.0 / n_in) if act == "silu" else np.sqrt(2.0 / (n_in + n_out))
        P += [rng.normal(0.0, std, (n_in, n_out)), np.zeros((1, n_out))]
    return P


def forward(X, P, act):
    f = ACT[act][0]
    Z1 = X @ P[0] + P[1]
    A1 = f(Z1)
    Z2 = A1 @ P[2] + P[3]
    A2 = f(Z2)
    Z3 = A2 @ P[4] + P[5]
    return Z1, A1, Z2, A2, Z3, sigmoid(Z3)


def backward(X, y, P, act):
    df = ACT[act][1]
    Z1, A1, Z2, A2, Z3, yh = forward(X, P, act)
    dZ3 = (yh - y) / len(y)
    dA2 = dZ3 @ P[4].T
    dZ2 = dA2 * df(Z2, A2)
    dA1 = dZ2 @ P[2].T
    dZ1 = dA1 * df(Z1, A1)
    G = [X.T @ dZ1, dZ1.sum(axis=0, keepdims=True), A1.T @ dZ2, dZ2.sum(axis=0, keepdims=True),
         A2.T @ dZ3, dZ3.sum(axis=0, keepdims=True)]
    return G, dict(Z1=Z1, A1=A1, Z2=Z2, A2=A2, Z3=Z3, yh=yh, dZ3=dZ3, dA2=dA2, dZ2=dZ2, dA1=dA1, dZ1=dZ1)


def new_state(P):
    return {"m": [np.zeros_like(p) for p in P], "v": [np.zeros_like(p) for p in P], "t": 0}


def sgd_step(P, G, S, lr):
    return [p - lr * g for p, g in zip(P, G)]


def rmsprop_step(P, G, S, lr):
    S["v"] = [RHO * v + (1 - RHO) * g ** 2 for v, g in zip(S["v"], G)]
    return [p - lr * g / (np.sqrt(v) + EPS) for p, g, v in zip(P, G, S["v"])]


def adam_step(P, G, S, lr):
    S["t"] += 1
    S["m"] = [BETA1 * m + (1 - BETA1) * g for m, g in zip(S["m"], G)]
    S["v"] = [BETA2 * v + (1 - BETA2) * g ** 2 for v, g in zip(S["v"], G)]
    c1, c2 = 1 - BETA1 ** S["t"], 1 - BETA2 ** S["t"]
    return [p - lr * (m / c1) / (np.sqrt(v / c2) + EPS) for p, m, v in zip(P, S["m"], S["v"])]


STEP = {"sgd": sgd_step, "rmsprop": rmsprop_step, "adam": adam_step}


def train(opt, act, D, P, epochs, lr):
    X, y, Xte, yte = D
    P0, S, loss = P, new_state(P), []
    for _ in range(epochs):
        G, c = backward(X, y, P, act)
        loss.append(bce(c["yh"], y))
        P = STEP[opt](P, G, S, lr)
    p_te = forward(Xte, P, act)[-1]
    return {"P0": P0, "P": P, "loss": loss, "p_te": p_te, "acc": acc(p_te, yte),
            "final": bce(forward(X, P, act)[-1], y)}


def gradient_check(act, eps=1e-6):
    rng = np.random.default_rng(0)
    X = rng.normal(size=(8, 4))
    y = (rng.random((8, 1)) > 0.5).astype(float)
    P = init_params(act, 3)
    P[1::2] = [rng.normal(0.0, 0.1, b.shape) for b in P[1::2]]
    G, _ = backward(X, y, P, act)
    worst = 0.0
    for p, g in zip(P, G):
        for i in np.ndindex(p.shape):
            old = p[i]
            p[i] = old + eps
            lp = bce(forward(X, P, act)[-1], y)
            p[i] = old - eps
            lm = bce(forward(X, P, act)[-1], y)
            p[i] = old
            worst = max(worst, abs((lp - lm) / (2 * eps) - g[i]))
    return worst


def run_pytorch(opt, act, P0, X, y, Xte, epochs, lr):
    try:
        import torch
        import torch.nn as nn
    except ImportError:
        return None
    torch.set_default_dtype(torch.float64)
    f = {"silu": nn.SiLU, "sigmoid": nn.Sigmoid, "tanh": nn.Tanh}[act]
    m = nn.Sequential(nn.Linear(4, 4), f(), nn.Linear(4, 4), f(), nn.Linear(4, 1), nn.Sigmoid())
    with torch.no_grad():
        for layer, W, b in zip((m[0], m[2], m[4]), P0[::2], P0[1::2]):
            layer.weight.copy_(torch.from_numpy(W.T.copy()))
            layer.bias.copy_(torch.from_numpy(b.ravel().copy()))
    if opt == "adam":
        o = torch.optim.Adam(m.parameters(), lr=lr, betas=(BETA1, BETA2), eps=EPS)
    elif opt == "rmsprop":
        o = torch.optim.RMSprop(m.parameters(), lr=lr, alpha=RHO, eps=EPS)
    else:
        o = torch.optim.SGD(m.parameters(), lr=lr)
    Xt, yt = torch.from_numpy(X), torch.from_numpy(y)
    for _ in range(epochs):
        o.zero_grad()
        nn.functional.binary_cross_entropy(m(Xt), yt).backward()
        o.step()
    with torch.no_grad():
        return m(torch.from_numpy(Xte)).numpy()


def run_sklearn(opt, act, X, y, Xte, epochs, lr):
    try:
        from sklearn.neural_network import MLPClassifier
    except ImportError:
        return None
    if act == "silu" or opt == "rmsprop":
        return None
    if opt == "adam":
        extra = dict(beta_1=BETA1, beta_2=BETA2, epsilon=EPS)
    else:
        extra = dict(momentum=0.0, nesterovs_momentum=False)
    m = MLPClassifier(hidden_layer_sizes=(4, 4), activation={"sigmoid": "logistic", "tanh": "tanh"}[act],
                      solver=opt, learning_rate_init=lr, alpha=0.0, batch_size=len(X), max_iter=epochs,
                      tol=0.0, n_iter_no_change=epochs + 1, shuffle=False, random_state=1, **extra)
    m.fit(X, y.ravel().astype(int))
    return m.predict_proba(Xte)[:, 1:2]

In [9]:
X, y = load_data()
D = split(X, y)
Xtr, ytr, Xte, yte = D
print("samples:", len(X), " train:", len(Xtr), " test:", len(Xte), " architecture:", SIZES)

print("\n[1] Gradient check: max |analytical - numerical|")
for a in ACTS:
    print(f"{a:8} {gradient_check(a):.1e}")

R = {}
for o in OPTS:
    for a in ACTS:
        R[o, a] = [train(o, a, D, init_params(a, s), EPOCHS, LR[o]) for s in SEEDS]

print("\n[2] Custom network, 5 seeds")
print("optimizer activation  train_loss  test_acc (mean +- std)  epochs_to_loss<0.05")
for (o, a), runs in R.items():
    loss = np.mean([r["loss"][-1] for r in runs])
    accs = [r["acc"] for r in runs]
    ep = np.mean([reached(r["loss"]) for r in runs])
    print(f"{o:9} {a:10} {loss:10.4f}  {np.mean(accs):.4f} +- {np.std(accs):.4f}  {ep:19.0f}")

print("\n[3] Same initial weights: custom vs PyTorch vs scikit-learn (test set)")
print("optimizer activation  library     acc     loss    max|p - p_custom| (pytorch)")
for (o, a), runs in R.items():
    r = runs[0]
    libs = {"custom": r["p_te"],
            "pytorch": run_pytorch(o, a, r["P0"], Xtr, ytr, Xte, EPOCHS, LR[o]),
            "sklearn": run_sklearn(o, a, Xtr, ytr, Xte, EPOCHS, LR[o])}
    for name, p in libs.items():
        if p is not None:
            d = f"{np.abs(p - r['p_te']).max():.1e}" if name == "pytorch" else "-"
            print(f"{o:9} {a:10} {name:9} {acc(p, yte):.4f}  {bce(p, yte):.4f}  {d}")

np.set_printoptions(precision=8, suppress=True)
act = "silu"
x, t = Xtr[:1], ytr[:1]
P0 = MAN_INIT
G, c = backward(x, t, P0, act)
names = ["W1", "B1", "W2", "B2", "W3", "B3"]

print("\n[4] Manual walkthrough: one sample, hidden activation", act)
print("x =", x[0], "  y =", t[0, 0])

print("\n1. Initial parameters")
for n, p in zip(names, P0):
    print(n, "=\n", p)

print("\n2. Forward propagation")
for k in ["Z1", "A1", "Z2", "A2", "Z3", "yh"]:
    print(k, "=", c[k][0])

print("\n3. Loss")
print("J = -[y*ln(y_hat) + (1-y)*ln(1-y_hat)] =", bce(c["yh"], t))

print("\n4. Backpropagation")
for k in ["dZ3", "dA2", "dZ2", "dA1", "dZ1"]:
    print(k, "=", c[k][0])

print("\n5. All gradients")
for n, g in zip(names, G):
    print("d" + n, "=\n", g)

print("\n6. Parameter updates")
print("SGD:     w = w - lr*g")
print("RMSprop: v = (1-rho)*g^2,  w = w - lr*g/(sqrt(v)+eps)")
print("Adam:    m = (1-b1)*g,  v = (1-b2)*g^2,  w = w - lr*(m/(1-b1))/(sqrt(v/(1-b2))+eps)")
print("W3 old =", P0[4].ravel(), " B3 old =", P0[5].ravel())
P1 = {}
for o in OPTS:
    S = new_state(P0)
    P1[o] = STEP[o](P0, G, S, MAN_LR[o])
    print(f"{o} (lr = {MAN_LR[o]}): W3 new =", P1[o][4].ravel(), " B3 new =", P1[o][5].ravel())
    if o == "adam":
        print("   m of W3 =", S["m"][4].ravel(), "\n   v of W3 =", S["v"][4].ravel())

print("\n7. After the first step")
print(f"before: y_hat = {c['yh'][0, 0]:.6f}  J = {bce(c['yh'], t):.6f}")
for o in OPTS:
    y1 = forward(x, P1[o], act)[-1]
    print(f"{o}: y_hat = {y1[0, 0]:.6f}  J = {bce(y1, t):.6f}")

print(f"\nFINAL RESULT AFTER {MAN_EPOCHS} EPOCHS (J)")
print("activation ", *[f"{o:>10}" for o in OPTS])
for a in ACTS:
    J = [train(o, a, (x, t, x, t), P0, MAN_EPOCHS, MAN_LR[o])["final"] for o in OPTS]
    print(f"{a:10}", *[f"{j:10.3e}" for j in J])

samples: 1372  train: 1030  test: 342  architecture: (4, 4, 4, 1)

[1] Gradient check: max |analytical - numerical|
silu     8.6e-11
sigmoid  1.2e-10
tanh     6.6e-11

[2] Custom network, 5 seeds
optimizer activation  train_loss  test_acc (mean +- std)  epochs_to_loss<0.05
adam      silu           0.0001  1.0000 +- 0.0000                  102
adam      sigmoid        0.0005  1.0000 +- 0.0000                  295
adam      tanh           0.0001  1.0000 +- 0.0000                   95
rmsprop   silu           0.0000  1.0000 +- 0.0000                  301
rmsprop   sigmoid        0.0000  1.0000 +- 0.0000                  695
rmsprop   tanh           0.0000  1.0000 +- 0.0000                  243
sgd       silu           0.0004  1.0000 +- 0.0000                   66
sgd       sigmoid        0.0076  0.9971 +- 0.0032                  422
sgd       tanh           0.0003  1.0000 +- 0.0000                   53

[3] Same initial weights: custom vs PyTorch vs scikit-learn (test set)
optimizer activ

In [15]:
import matplotlib


def _in_notebook():
    try:
        from IPython import get_ipython
        return get_ipython() is not None
    except ImportError:
        return False


IN_NOTEBOOK = _in_notebook()
if not IN_NOTEBOOK:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

LABEL = {"adam": "Adam", "rmsprop": "RMSprop", "sgd": "SGD", "silu": "SiLU", "sigmoid": "Sigmoid", "tanh": "Tanh"}
SURFACE, INK, INK2, MUTED, GRID, AXIS = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
BLUE, ORANGE, AQUA, YELLOW = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"
ACT_STYLE = {"silu": (BLUE, "-"), "sigmoid": (ORANGE, "--"), "tanh": (AQUA, ":")}
LOSS_LABEL = "Loss (BCE)"
SEQ_CMAP = LinearSegmentedColormap.from_list("seq_blue", ["#cde2fb", "#256abf", "#104281"])

plt.rcParams.update({"font.family": "DejaVu Sans", "figure.facecolor": SURFACE})


def style_axes(ax, title, xlabel, ylabel, grid_axis="y"):
    ax.set_facecolor(SURFACE)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color(AXIS)
    ax.tick_params(colors=MUTED, labelsize=9)
    if grid_axis != "none":
        ax.grid(axis=grid_axis, color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    ax.set_title(title, loc="left", color=INK, fontsize=11, fontweight="bold")
    ax.set_xlabel(xlabel, color=INK2, fontsize=9.5)
    ax.set_ylabel(ylabel, color=INK2, fontsize=9.5)


def end_labels(ax, items, min_gap=0.07):
    inv = ax.transAxes.inverted()
    xmax = ax.get_xlim()[1]
    pos = [min(max(inv.transform(ax.transData.transform((xmax, y)))[1], 0.03), 0.97)
           for y, _, _ in items]
    order = np.argsort(pos)
    fixed = list(pos)
    for k in range(1, len(order)):
        fixed[order[k]] = max(fixed[order[k]], fixed[order[k - 1]] + min_gap)
    over = max(fixed) - 0.97
    if over > 0:
        fixed = [p - over for p in fixed]
    for p, (_, text, color) in zip(fixed, items):
        ax.plot([1.015, 1.05], [p, p], transform=ax.transAxes, color=color, lw=2.2,
                solid_capstyle="round", clip_on=False)
        ax.text(1.06, p, text, transform=ax.transAxes, va="center", color=INK2, fontsize=9)


def legend(ax, **kw):
    ax.legend(frameon=False, fontsize=9, labelcolor=INK2, **kw)


def save(fig, name):
    plt.show()
    plt.close(fig)


def plot_activations(results, seeds):
    fig, ax = plt.subplots(1, 3, figsize=(17, 4.8))
    fig.subplots_adjust(wspace=0.5)

    for act, r in results.items():
        color, ls = ACT_STYLE[act]
        h = r["runs"][0]["hist"]
        ax[0].semilogy(h["loss"], color=color, ls=ls, lw=1.8, label=LABEL[act])
        ax[1].semilogx(np.arange(1, len(h["test_acc"]) + 1), h["test_acc"],
                       color=color, ls=ls, lw=1.8, label=LABEL[act])
    ax[1].set_xlim(1, EPOCHS)
    ax[1].set_ylim(0.98, 1.002)
    style_axes(ax[0], "Loss vs. epoch (training set)", "Epoch", LOSS_LABEL)
    style_axes(ax[1], "Test accuracy (zoomed: 0.98-1.0)", "Epoch (log scale)",
               "Accuracy (test)")
    legend(ax[0], loc="upper right")
    legend(ax[1], loc="lower right")
    end_labels(ax[0], [(r["runs"][0]["hist"]["loss"][-1], LABEL[a], ACT_STYLE[a][0])
                       for a, r in results.items()])
    end_labels(ax[1], [(r["runs"][0]["hist"]["test_acc"][-1], LABEL[a], ACT_STYLE[a][0])
                       for a, r in results.items()])

    for i, (act, r) in enumerate(results.items()):
        accs = [run["test_acc"] for run in r["runs"]]
        jitter = np.linspace(-0.18, 0.18, len(accs))
        ax[2].plot(i + jitter, accs, "o", color=ACT_STYLE[act][0], ms=8, mec=SURFACE, mew=1.5)
        ax[2].hlines(np.mean(accs), i - 0.28, i + 0.28, color=INK2, lw=1.8)
    ax[2].set_xticks(range(len(results)))
    ax[2].set_xticklabels([LABEL[a] for a in results])
    style_axes(ax[2], f"Spread over {len(seeds)} seeds (line = mean)", "", "Accuracy (test)")
    save(fig, "fig1_activations.png")


def plot_libraries(best, results, libs, n_te):
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.8))
    fig.subplots_adjust(wspace=0.55)
    L = libs[best]
    series = [("Custom (NumPy)", results[best]["runs"][0]["hist"]["loss"], BLUE, "-", 1.8),
              ("PyTorch", L.get("torch", {}).get("hist"), ORANGE, "--", 1.8),
              ("scikit-learn", L.get("sklearn", {}).get("hist"), AQUA, ":", 2.4)]
    series = [s for s in series if s[1] is not None]
    for name, h, color, ls, lw in series:
        ax[0].semilogy(h, color=color, ls=ls, lw=lw, label=name)
    style_axes(ax[0], f"Loss vs. epoch, {LABEL[best]}", "Epoch", LOSS_LABEL)
    legend(ax[0], loc="center right")
    ax[0].text(0.02, 0.04, "scikit-learn starts from a different weight initialization",
               transform=ax[0].transAxes, color=MUTED, fontsize=8.5)
    end_labels(ax[0], [(h[-1], name, color) for name, h, color, _, _ in series])

    impls = [("own", "Custom (NumPy)", BLUE, "o"), ("torch", "PyTorch", ORANGE, "s"),
             ("sklearn", "scikit-learn", AQUA, "^")]
    for j, (key, name, color, marker) in enumerate(impls):
        xs, ys = [], []
        for i, act in enumerate(results):
            if key in libs[act]:
                xs.append(i + (j - 1) * 0.22)
                ys.append(libs[act][key]["test_acc"])
        ax[1].plot(xs, ys, marker, color=color, ms=9, mec=SURFACE, mew=1.5, label=name)
    ax[1].set_xticks(range(len(results)))
    ax[1].set_xticklabels([LABEL[a] for a in results])
    ax[1].set_ylim(0.985, 1.003)
    style_axes(ax[1], "Test accuracy: custom network vs. libraries", "", "Accuracy (test)")
    ax[1].text(0.99, 0.02, f"1 test error = {1 / n_te:.4f}", transform=ax[1].transAxes,
               ha="right", color=MUTED, fontsize=8.5)
    for i, act in enumerate(results):
        if "sklearn" not in libs[act]:
            ax[1].text(i + 0.22, 1.0, "not in\nsklearn", ha="center", va="center", color=MUTED, fontsize=8)
    legend(ax[1], loc="lower left")
    save(fig, "fig2_libraries.png")


def plot_predictions(net, X_te, y_te, act):
    p = net.predict(X_te).ravel()
    yt = y_te.ravel().astype(int)
    order = np.lexsort((p, yt))
    p_s, y_s = p[order], yt[order]
    x = np.arange(len(p_s))

    fig, ax = plt.subplots(1, 2, figsize=(13, 4.8))
    fig.subplots_adjust(wspace=0.3)
    for c, color in ((0, BLUE), (1, ORANGE)):
        m = y_s == c
        ax[0].plot(x[m], p_s[m], "o", color=color, ms=5, mec=SURFACE, mew=0.6, label=f"class {c}")
    ax[0].axhline(0.5, color=MUTED, lw=1.2, ls="--")
    n0 = int((y_s == 0).sum())
    ax[0].text(n0 / 2, 0.07, f"class 0 (n={n0})", color=INK2, ha="center", va="bottom", fontsize=9)
    ax[0].text(n0 + (len(x) - n0) / 2, 0.93, f"class 1 (n={len(x) - n0})", color=INK2,
               ha="center", va="top", fontsize=9)
    ax[0].text(len(x) - 1, 0.52, "threshold 0.5", color=MUTED, ha="right", fontsize=8.5)
    style_axes(ax[0], f"Predictions vs. targets (test, {LABEL[act]})",
               "Test samples (sorted by class and probability)", "Predicted probability of class 1")
    legend(ax[0], loc="upper left")

    cm = np.zeros((2, 2), dtype=int)
    for t, q in zip(yt, (p > 0.5).astype(int)):
        cm[t, q] += 1
    ax[1].imshow(cm, cmap=SEQ_CMAP, vmin=0, vmax=cm.max())
    for i in range(2):
        for j in range(2):
            ax[1].text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=16,
                       color="#ffffff" if cm[i, j] > cm.max() / 2 else INK)
    ax[1].set_xticks([0, 1])
    ax[1].set_yticks([0, 1])
    style_axes(ax[1], "Confusion matrix (test)", "Predicted class", "True class", grid_axis="none")
    ax[1].tick_params(length=0)
    for s in ax[1].spines.values():
        s.set_visible(False)
    save(fig, "fig3_predictions.png")

TypeError: plot_predictions() missing 2 required positional arguments: 'y_te' and 'act'